# 22 - Champions League: model setups and model type

P5-01 put the Champions League in the pipeline: CL rows in `all_matches.csv`
with 90-minute results, shared-scale `EloDiff` (notebook 21's E2a+E2b) and
competition-scoped form. This notebook chooses **how CL matches are predicted**
(docs/multi-league-plan.md §4B.7-4B.8, D5):

| Setup | What |
|---|---|
| **(a)** | The shipped pooled domestic LR applied as it is; CL rows get the baseline league's indicators (all 0). Zero new parameters. |
| **(b)** | The pooled LR trained on domestic **plus** European rows, with `Comp_*`, `Knockout`, `EloDiff×Euro`, `XGDDiffLast5×Euro`; optionally `Leg2AggDiff`. |
| **(c)** | An LR on European rows only, same features. |
| **(d)** | Trees / boosting on (b)'s rows and features (nb 18 hyperparameters), sigmoid-calibrated. |
| **(e)** | Two-stage: (a)'s logits plus a shallow tree (depth 2, ≥40 per leaf) fitted on European rows for the residual. |

**Evaluation** (plan §5): walk-forward by season, test 22-23 to 25-26, log loss
first, paired bootstrap (5,000 resamples, seed 42). On the CL alone (628
matches) and on CL + Europa League + Conference League (D10, ~2,000) for power.

**Rules.** (b) or (c) ships over (a) only if its CI against (a) is entirely
below 0 (*better*). A non-LR model (d)/(e) ships only if it beats the **best LR
setup** with a CI entirely below 0 **and** is better in most test seasons
(§4B.8); otherwise the model type stays LR (D5). If (b) shares training data
with the domestic leagues it must also be non-inferior on domestic matches.

A neutral venue (the final, relocated home ties) is scored both ways round and
averaged, as the pipeline does (§4B.4). Every feature is a home-minus-away
difference, so the reverse orientation is the negated features with H and A
swapped.

In [1]:
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # the pipeline reads data/ relative to the repo root
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from european import EUROPEAN, results_for_elo
from evaluate import better, non_inferior, paired_bootstrap, per_match_log_loss, seasons_better
from features import build_history
from load_data import build_cup_rows, read_matches
from model import FEATURES, LEAGUE_COLUMNS, OWN_MODEL, league_columns, make_model

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

TEST = ["22-23", "23-24", "24-25", "25-26"]
LAST = TEST[-1]          # 26-27 is unfinished: never a test or training row here
LABELS = ["A", "D", "H"]  # model.classes_ order

## Data

The pipeline's own history: the 7 leagues and the CL from `all_matches.csv`
(P5-01), plus the Europa League and Conference League rows built the same way
from P4-01's caches. `build_history` gives every row its features exactly as
the pipeline would: league rows are unchanged by the European rows, European
rows get both clubs' shared-scale rating going in, and form is scoped to the
competition (S3 for European rows - P5-03 tests the other scopes).

In [2]:
matches = read_matches(cups=True)
domestic = matches[~matches["Div"].isin(EUROPEAN)]
el_ecl = build_cup_rows(domestic, ["EL", "ECL"])

history = pd.concat([matches, el_ecl.reindex(columns=matches.columns)], ignore_index=True)
history = history[history["Season"] <= LAST]

data, _, _ = build_history(history, european=results_for_elo(domestic))
data["Euro"] = data["Div"].isin(EUROPEAN).astype(int)

print(data.groupby(["Div", "Season"]).size().unstack().fillna(0).astype(int).to_string())

Season  21-22  22-23  23-24  24-25  25-26
Div                                      
CL        125    125    125    189    189
D1        306    306    306    306    306
E0        380    380    380    380    380
ECL       140    141    141    153    153
EL        139    141    141    189    189
F1        380    380    306    306    306
I1        380    380    380    380    380
N1        306    306    306    306    306
P1        306    306    306    306    306
SP1       380    380    380    380    380


European features. `Knockout` = a knockout tie (play-off round to the final).
`Leg2AggDiff` = the leg-2 home side's lead from leg 1 (its leg-1 away goals
minus home goals), clipped to ±2, 0 on every other match (§4B.3). The
`×Euro` interactions let European matches weight Elo and form differently.

In [3]:
KNOCKOUT = {"KO_PLAYOFF", "R16", "QF", "SF", "FINAL"}

data["Knockout"] = data["Stage"].isin(KNOCKOUT).astype(int)
for div in EUROPEAN:
    data[f"Comp_{div}"] = (data["Div"] == div).astype(int)
data["EloDiff_x_Euro"] = data["EloDiff"] * data["Euro"]
data["XGD_x_Euro"] = data["XGDDiffLast5"] * data["Euro"]

legs = data[data["Euro"] == 1]
first_legs = legs[legs["Leg"] == 1].set_index(["Div", "Season", "Stage", "AwayTeam", "HomeTeam"])
second = data.index[(data["Euro"] == 1) & (data["Leg"] == 2)]
leg1 = first_legs.reindex(pd.MultiIndex.from_frame(
    data.loc[second, ["Div", "Season", "Stage", "HomeTeam", "AwayTeam"]]
))
lead = (leg1["FTAG"] - leg1["FTHG"]).to_numpy()  # leg-2 home side = leg-1 away side

data["Leg1Lead"] = np.nan
data.loc[second, "Leg1Lead"] = lead
data["Leg2AggDiff"] = data["Leg1Lead"].clip(-2, 2).fillna(0)

print(f"second legs: {len(second)}, leg 1 found for {np.isfinite(lead).sum()}")

data = league_columns(data)
INDICATORS = [c for c in LEAGUE_COLUMNS.values() if c in data.columns]
EURO_TERMS = [f"Comp_{d}" for d in EUROPEAN] + ["Knockout", "EloDiff_x_Euro", "XGD_x_Euro"]

# Columns that flip sign when home and away swap (for the neutral venue)
ANTISYMMETRIC = ["EloDiff", "XGDDiffLast5", "GoalsPerGameDiff", "EloDiff_x_Euro", "XGD_x_Euro", "Leg2AggDiff"]

euro_test = data[(data["Euro"] == 1) & data["Season"].isin(TEST)]
cl_test = euro_test[euro_test["Div"] == "CL"]
print(f"test matches: CL {len(cl_test)}, CL+EL+ECL {len(euro_test)}, "
      f"neutral {int(euro_test['Neutral'].sum())} (finals {int((euro_test['Stage'] == 'FINAL').sum())})")
print(euro_test.groupby("Div")["FTR"].value_counts(normalize=True).unstack().round(3))

second legs: 305, leg 1 found for 305
test matches: CL 628, CL+EL+ECL 1876, neutral 69 (finals 12)
FTR      A      D      H
Div                     
CL   0.323  0.182  0.495
ECL  0.311  0.218  0.471
EL   0.294  0.208  0.498


## Walk-forward

For each test season `s`, a fresh model is fitted on every earlier season
(21-22 on) and predicts the European matches of `s`. `train_rows` says which
rows a setup trains on.

In [4]:
DOMESTIC = data["Euro"] == 0
EURO = data["Euro"] == 1
CL_ROWS = data["Div"] == "CL"


def predict(model, frame, features):
    """H/D/A probabilities (A, D, H columns), neutral venues averaged both ways."""

    proba = pd.DataFrame(model.predict_proba(frame[features]), index=frame.index, columns=model.classes_)[LABELS]

    neutral = frame["Neutral"].fillna(0).astype(int) == 1
    if neutral.any():
        flipped = frame.loc[neutral, features].copy()
        flipped[[c for c in ANTISYMMETRIC if c in features]] *= -1
        reverse = pd.DataFrame(model.predict_proba(flipped), index=flipped.index, columns=model.classes_)
        proba.loc[neutral, "H"] = (proba.loc[neutral, "H"] + reverse["A"]) / 2
        proba.loc[neutral, "D"] = (proba.loc[neutral, "D"] + reverse["D"]) / 2
        proba.loc[neutral, "A"] = (proba.loc[neutral, "A"] + reverse["H"]) / 2

    return proba


def walk(make, features, train_rows, test_rows=EURO):
    """Out-of-sample probabilities for ``test_rows`` in the test seasons."""

    out = []
    for season in TEST:
        train = data[train_rows & (data["Season"] < season)]
        test = data[test_rows & (data["Season"] == season)]
        model = make().fit(train[features], train["FTR"])
        out.append(predict(model, test, features))
    return pd.concat(out).loc[data.index[test_rows & data["Season"].isin(TEST)]]


calibrated = lambda clf: CalibratedClassifierCV(estimator=clf, method="sigmoid", cv=5)

## Setups (a)-(c): logistic regression

In [5]:
A_FEATURES = FEATURES + INDICATORS
B_FEATURES = A_FEATURES + EURO_TERMS
# European rows only: the CL is the baseline competition, and the x Euro
# terms would just repeat EloDiff / XGDDiffLast5
C_FEATURES = FEATURES + ["Comp_EL", "Comp_ECL", "Knockout"]
C_CL_FEATURES = FEATURES + ["Knockout"]

proba = {}
# (a) the shipped pooled model: every league match (E0 included), indicators
proba["(a) domestic LR as is"] = walk(make_model, A_FEATURES, DOMESTIC)
# (b) + European rows and terms; b-CL adds the CL rows only, as the plan wrote it
proba["(b) domestic + Euro rows"] = walk(make_model, B_FEATURES, DOMESTIC | EURO)
proba["(b-CL) domestic + CL rows"] = walk(make_model, B_FEATURES, DOMESTIC | CL_ROWS)
proba["(b+leg2) with Leg2AggDiff"] = walk(make_model, B_FEATURES + ["Leg2AggDiff"], DOMESTIC | EURO)
# (c) European rows only; c-CL the CL rows only
proba["(c) Euro-only LR"] = walk(make_model, C_FEATURES, EURO)
proba["(c+leg2) with Leg2AggDiff"] = walk(make_model, C_FEATURES + ["Leg2AggDiff"], EURO)
proba["(c-CL) CL-only LR"] = walk(make_model, C_CL_FEATURES, CL_ROWS)

Benchmark: each test season's European matches at the 90-minute H/D/A rates of
the earlier seasons' European matches, by phase (league/group phase or
knockout).

In [6]:
def base_rates():
    out = []
    for season in TEST:
        train = data[EURO & (data["Season"] < season)]
        test = data[EURO & (data["Season"] == season)]
        rates = train.groupby("Knockout")["FTR"].value_counts(normalize=True).unstack()[LABELS]
        out.append(pd.DataFrame(rates.loc[test["Knockout"]].to_numpy(), index=test.index, columns=LABELS))
    return pd.concat(out).loc[euro_test.index]


proba["base rates by phase"] = base_rates()

## Setups (d) and (e): trees and a two-stage model

(d): nb 18's hyperparameters, on (b)'s rows and features. scikit-learn's three
need nothing new; XGBoost is in `.venv` but not `requirements.txt` (§4B.8 point 5).

(e): stage 1 is (a). Stage 2, fitted on the earlier seasons' European rows only:
a depth-2 tree (≥ 40 matches per leaf) on knockout / leg state / venue /
rating gap picks out cells, and a multinomial LR on (a)'s two log-odds plus the
leaf indicators corrects (a) in each cell. Stage-1 probabilities for those rows
are out-of-sample: (a) never trains on European rows.

In [7]:
TREES = {
    "(d) random forest": lambda: calibrated(RandomForestClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=20, max_features="sqrt", n_jobs=-1, random_state=42)),
    "(d) extra trees": lambda: calibrated(ExtraTreesClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=20, max_features="sqrt", n_jobs=-1, random_state=42)),
    "(d) hist gradient boosting": lambda: calibrated(HistGradientBoostingClassifier(
        learning_rate=0.03, max_iter=200, max_depth=3, min_samples_leaf=40, l2_regularization=1.0, random_state=42)),
    "(d) XGBoost": lambda: calibrated(XGBClassifier(
        n_estimators=300, learning_rate=0.02, max_depth=2, min_child_weight=10, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=5.0, objective="multi:softprob", eval_metric="mlogloss",
        n_jobs=-1, random_state=42)),
}


class Encoded:
    """XGBoost wants integer classes; give it A/D/H -> 0/1/2 and back."""

    def __init__(self, make):
        self.make = make

    def fit(self, X, y):
        self.model = self.make().fit(X, pd.Index(LABELS).get_indexer(y))
        self.classes_ = np.array(LABELS)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


for name, make in TREES.items():
    wrapped = (lambda make=make: Encoded(make)) if "XGBoost" in name else make
    proba[name] = walk(wrapped, B_FEATURES, DOMESTIC | EURO)

In [8]:
TREE_INPUTS = ["EloDiff", "Knockout", "Leg2AggDiff", "Neutral", "XGDDiffLast5"]


def two_stage():
    out = []
    for season in TEST:
        stage1 = make_model().fit(data.loc[DOMESTIC & (data["Season"] < season), A_FEATURES],
                                  data.loc[DOMESTIC & (data["Season"] < season), "FTR"])
        train = data[EURO & (data["Season"] < season)]
        test = data[EURO & (data["Season"] == season)]

        def logits(frame):
            p = predict(stage1, frame, A_FEATURES).clip(1e-6)
            return np.column_stack([np.log(p["H"] / p["A"]), np.log(p["D"] / p["A"])])

        tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=40, random_state=42)
        tree.fit(train[TREE_INPUTS], train["FTR"])
        leaves = np.unique(tree.apply(train[TREE_INPUTS]))

        def design(frame):
            leaf = tree.apply(frame[TREE_INPUTS])
            return np.column_stack([logits(frame)] + [(leaf == l).astype(int) for l in leaves[1:]])

        stage2 = LogisticRegression(max_iter=2000).fit(design(train), train["FTR"])
        out.append(pd.DataFrame(stage2.predict_proba(design(test)), index=test.index, columns=stage2.classes_)[LABELS])
    return pd.concat(out).loc[euro_test.index]


proba["(e) LR + depth-2 tree residual"] = two_stage()

## Results

Log loss on the CL test matches and on all European test matches; each setup
against (a), paired bootstrap of the per-match difference (negative = better
than (a)), and the number of test seasons it beats (a) in.

In [9]:
REFERENCE = "(a) domestic LR as is"


def loss(name, rows):
    return per_match_log_loss(rows["FTR"], proba[name].loc[rows.index][LABELS], LABELS)


def table(rows, reference=REFERENCE):
    ref = loss(reference, rows)
    out = []
    for name in proba:
        cand = loss(name, rows)
        boot = paired_bootstrap(cand - ref)
        out.append({
            "setup": name,
            "log loss": cand.mean(),
            "accuracy %": 100 * (proba[name].loc[rows.index][LABELS].idxmax(axis=1) == rows["FTR"]).mean(),
            "diff vs ref": boot["mean"],
            "CI low": boot["ci"][0],
            "CI high": boot["ci"][1],
            "better": better(boot["ci"]),
            "seasons better": seasons_better(rows["FTR"], proba[name].loc[rows.index][LABELS],
                                             proba[reference].loc[rows.index][LABELS], rows["Season"], LABELS),
            "SD of diff": (cand - ref).std(ddof=1),
        })
    return pd.DataFrame(out).set_index("setup")


cl_table = table(cl_test)
euro_table = table(euro_test)
print(f"CL, {len(cl_test)} matches")
print(cl_table.round(4).to_string())
print(f"\nCL + EL + ECL, {len(euro_test)} matches")
print(euro_table.round(4).to_string())

CL, 628 matches
                                log loss  accuracy %  diff vs ref  CI low  CI high  better  seasons better  SD of diff
setup                                                                                                                 
(a) domestic LR as is             0.9324     57.1656       0.0000  0.0000   0.0000   False               0      0.0000
(b) domestic + Euro rows          0.9266     57.6433      -0.0058 -0.0121   0.0005   False               2      0.0829
(b-CL) domestic + CL rows         0.9345     56.8471       0.0021 -0.0052   0.0099   False               2      0.0994
(b+leg2) with Leg2AggDiff         0.9310     57.6433      -0.0014 -0.0097   0.0075   False               2      0.1105
(c) Euro-only LR                  0.9236     58.1210      -0.0088 -0.0177   0.0001   False               2      0.1140
(c+leg2) with Leg2AggDiff         0.9275     57.9618      -0.0049 -0.0151   0.0058   False               2      0.1325
(c-CL) CL-only LR               

Per season, CL only:

In [10]:
per_season = pd.DataFrame({
    name: [log_loss(cl_test.loc[cl_test["Season"] == s, "FTR"],
                    proba[name].loc[cl_test.index[cl_test["Season"] == s]][LABELS], labels=LABELS) for s in TEST]
    for name in proba
}, index=TEST).T
print(per_season.round(4).to_string())

                                 22-23   23-24   24-25   25-26
(a) domestic LR as is           0.9135  0.9216  0.9115  0.9730
(b) domestic + Euro rows        0.9182  0.9238  0.9002  0.9603
(b-CL) domestic + CL rows       0.9316  0.9376  0.9046  0.9644
(b+leg2) with Leg2AggDiff       0.9323  0.9238  0.9002  0.9656
(c) Euro-only LR                0.9178  0.9237  0.8928  0.9584
(c+leg2) with Leg2AggDiff       0.9274  0.9221  0.8937  0.9650
(c-CL) CL-only LR               0.9278  0.9422  0.8911  0.9590
base rates by phase             1.0459  1.0457  1.0089  1.0247
(d) random forest               0.9110  0.9169  0.9103  0.9666
(d) extra trees                 0.9090  0.9143  0.9023  0.9714
(d) hist gradient boosting      0.9405  0.9232  0.9073  0.9754
(d) XGBoost                     0.9128  0.9229  0.9099  0.9642
(e) LR + depth-2 tree residual  0.8999  0.8832  0.9007  0.9545


### Honest selection: choose on 22-23/23-24, report on 24-25/25-26

Several LR variants were tried, so picking the best on all four test seasons
flatters it. As in nb 19/21: the best LR setup on the **tune** seasons'
European matches, then its CI against (a) on the **report** seasons only.

In [11]:
TUNE, REPORT = TEST[:2], TEST[2:]
LR_SETUPS = [n for n in proba if n.startswith(("(a)", "(b", "(c"))]

tune_rows = euro_test[euro_test["Season"].isin(TUNE)]
tune_loss = pd.Series({n: loss(n, tune_rows).mean() for n in LR_SETUPS}).sort_values()
print("tune seasons, CL+EL+ECL log loss:"); print(tune_loss.round(4).to_string())
chosen_on_tune = tune_loss.index[0]

for rows, label in [(cl_test, "CL"), (euro_test, "CL+EL+ECL")]:
    report_rows = rows[rows["Season"].isin(REPORT)]
    print()
    print(f"report seasons, {label} ({len(report_rows)} matches), vs (a):")
    print(table(report_rows).loc[LR_SETUPS, ["log loss", "diff vs ref", "CI low", "CI high", "seasons better"]].round(4).to_string())

tune seasons, CL+EL+ECL log loss:
(c+leg2) with Leg2AggDiff    0.9503
(c) Euro-only LR             0.9503
(a) domestic LR as is        0.9521
(b) domestic + Euro rows     0.9530
(b+leg2) with Leg2AggDiff    0.9536
(c-CL) CL-only LR            0.9590
(b-CL) domestic + CL rows    0.9676

report seasons, CL (378 matches), vs (a):


                           log loss  diff vs ref  CI low  CI high  seasons better
setup                                                                            
(a) domestic LR as is        0.9422       0.0000  0.0000   0.0000               0
(b) domestic + Euro rows     0.9302      -0.0120 -0.0214  -0.0026               2
(b-CL) domestic + CL rows    0.9345      -0.0077 -0.0161   0.0008               2
(b+leg2) with Leg2AggDiff    0.9329      -0.0093 -0.0211   0.0028               2
(c) Euro-only LR             0.9256      -0.0167 -0.0286  -0.0051               2
(c+leg2) with Leg2AggDiff    0.9294      -0.0129 -0.0261   0.0008               2
(c-CL) CL-only LR            0.9250      -0.0172 -0.0302  -0.0042               2

report seasons, CL+EL+ECL (1062 matches), vs (a):


                           log loss  diff vs ref  CI low  CI high  seasons better
setup                                                                            
(a) domestic LR as is        0.9673       0.0000  0.0000   0.0000               0
(b) domestic + Euro rows     0.9607      -0.0066 -0.0126  -0.0008               2
(b-CL) domestic + CL rows    0.9670      -0.0002 -0.0045   0.0041               2
(b+leg2) with Leg2AggDiff    0.9566      -0.0106 -0.0179  -0.0035               2
(c) Euro-only LR             0.9557      -0.0116 -0.0190  -0.0044               2
(c+leg2) with Leg2AggDiff    0.9514      -0.0158 -0.0243  -0.0075               2
(c-CL) CL-only LR            0.9540      -0.0132 -0.0212  -0.0052               2


### Leg2AggDiff: does the leg-1 lead add to its setup?

In [12]:
for with_leg2, without in [("(b+leg2) with Leg2AggDiff", "(b) domestic + Euro rows"),
                           ("(c+leg2) with Leg2AggDiff", "(c) Euro-only LR")]:
    for rows, label in [(cl_test, "CL"), (euro_test, "CL+EL+ECL")]:
        boot = paired_bootstrap(loss(with_leg2, rows) - loss(without, rows))
        print(f"{with_leg2} - {without}, {label}: {boot['mean']:+.4f} "
              f"[{boot['ci'][0]:+.4f}, {boot['ci'][1]:+.4f}]")

(b+leg2) with Leg2AggDiff - (b) domestic + Euro rows, CL: +0.0044 [-0.0007, +0.0100]
(b+leg2) with Leg2AggDiff - (b) domestic + Euro rows, CL+EL+ECL: -0.0020 [-0.0054, +0.0013]


(c+leg2) with Leg2AggDiff - (c) Euro-only LR, CL: +0.0039 [-0.0009, +0.0091]


(c+leg2) with Leg2AggDiff - (c) Euro-only LR, CL+EL+ECL: -0.0024 [-0.0055, +0.0006]


## The best LR setup and the §4B.8 rule for (d)/(e)

In [13]:
NON_LR = [n for n in proba if n.startswith(("(d)", "(e)"))]

best_lr = cl_table.loc[LR_SETUPS, "log loss"].idxmin()
print("best LR setup on CL log loss:", best_lr)

for rows, label in [(cl_test, "CL"), (euro_test, "CL+EL+ECL")]:
    vs_best = table(rows, reference=best_lr).loc[NON_LR, ["log loss", "diff vs ref", "CI low", "CI high", "seasons better"]]
    vs_best["passes §4B.8"] = (vs_best["CI high"] < 0) & (vs_best["seasons better"] >= 3)
    print(f"\nnon-LR vs {best_lr}, {label}")
    print(vs_best.round(4).to_string())

best LR setup on CL log loss: (c) Euro-only LR



non-LR vs (c) Euro-only LR, CL
                                log loss  diff vs ref  CI low  CI high  seasons better  passes §4B.8
setup                                                                                               
(d) random forest                 0.9287       0.0050 -0.0061   0.0169               2         False
(d) extra trees                   0.9268       0.0032 -0.0095   0.0162               2         False
(d) hist gradient boosting        0.9376       0.0139  0.0021   0.0258               1         False
(d) XGBoost                       0.9294       0.0058 -0.0046   0.0164               2         False
(e) LR + depth-2 tree residual    0.9132      -0.0104 -0.0226   0.0022               3         False



non-LR vs (c) Euro-only LR, CL+EL+ECL
                                log loss  diff vs ref  CI low  CI high  seasons better  passes §4B.8
setup                                                                                               
(d) random forest                 0.9596       0.0062 -0.0003   0.0129               1         False
(d) extra trees                   0.9580       0.0046 -0.0020   0.0116               1         False
(d) hist gradient boosting        0.9666       0.0133  0.0062   0.0202               0         False
(d) XGBoost                       0.9597       0.0064  0.0002   0.0125               1         False
(e) LR + depth-2 tree residual    0.9538       0.0004 -0.0068   0.0077               1         False


## Gate 2: domestic non-inferiority for (b)

(b) trains the pooled model on European rows too, so the pooled leagues'
predictions change (E0 keeps its own model, untouched). Domestic test matches
of the pooled leagues, (b)'s model against the shipped pooled model:

In [14]:
POOLED_DOMESTIC = DOMESTIC & ~data["Div"].isin(OWN_MODEL)
dom_a = walk(make_model, A_FEATURES, DOMESTIC, test_rows=POOLED_DOMESTIC)
dom_b = walk(make_model, B_FEATURES, DOMESTIC | EURO, test_rows=POOLED_DOMESTIC)
dom_rows = data[POOLED_DOMESTIC & data["Season"].isin(TEST)]
dom_diff = (per_match_log_loss(dom_rows["FTR"], dom_b[LABELS], LABELS)
            - per_match_log_loss(dom_rows["FTR"], dom_a[LABELS], LABELS))
dom_boot = paired_bootstrap(dom_diff)
print(f"{len(dom_rows)} domestic matches (pooled leagues): (b) - (a) = {dom_boot['mean']:+.5f}, "
      f"CI [{dom_boot['ci'][0]:+.5f}, {dom_boot['ci'][1]:+.5f}], non-inferior: {non_inferior(dom_boot['ci'])}")

8010 domestic matches (pooled leagues): (b) - (a) = -0.00009, CI [-0.00056, +0.00037], non-inferior: True


## Power: the measured per-match SD

The plan's table (§4B.8) used SDs backed out of nb 18: **0.069** for an LR
variant against LR, **0.150** for a tree against LR. Measured here on European
matches. 95% CI half-width = 1.96·SD/√n; minimum detectable effect at 80%
power = 2.80·SD/√n; n needed to detect 0.005 = (2.80·SD/0.005)².

In [15]:
best_tree = cl_table.loc[[n for n in NON_LR if n.startswith("(d)")], "log loss"].idxmin()
sd = {
    "LR variant vs LR ((b) vs (a))": {
        "CL": cl_table.loc["(b) domestic + Euro rows", "SD of diff"],
        "CL+EL+ECL": euro_table.loc["(b) domestic + Euro rows", "SD of diff"]},
    f"Tree vs LR ({best_tree} vs (a))": {
        "CL": cl_table.loc[best_tree, "SD of diff"],
        "CL+EL+ECL": euro_table.loc[best_tree, "SD of diff"]},
    "Two-stage vs LR ((e) vs (a))": {
        "CL": cl_table.loc["(e) LR + depth-2 tree residual", "SD of diff"],
        "CL+EL+ECL": euro_table.loc["(e) LR + depth-2 tree residual", "SD of diff"]},
}

rows = []
for comparison, by_set in sd.items():
    for test_set, s in by_set.items():
        n = len(cl_test) if test_set == "CL" else len(euro_test)
        rows.append({
            "comparison": comparison, "measured on": test_set, "SD": s, "n": n,
            "95% CI ±": 1.96 * s / np.sqrt(n), "MDE (80%)": 2.80 * s / np.sqrt(n),
            "n for 0.005": int(round((2.80 * s / 0.005) ** 2)),
        })
power = pd.DataFrame(rows)
print(power.round(4).to_string(index=False))

                         comparison measured on     SD    n  95% CI ±  MDE (80%)  n for 0.005
      LR variant vs LR ((b) vs (a))          CL 0.0829  628    0.0065     0.0093         2157
      LR variant vs LR ((b) vs (a))   CL+EL+ECL 0.0944 1876    0.0043     0.0061         2796
Tree vs LR ((d) extra trees vs (a))          CL 0.1512  628    0.0118     0.0169         7166
Tree vs LR ((d) extra trees vs (a))   CL+EL+ECL 0.1381 1876    0.0062     0.0089         5978
       Two-stage vs LR ((e) vs (a))          CL 0.1590  628    0.0124     0.0178         7925
       Two-stage vs LR ((e) vs (a))   CL+EL+ECL 0.1652 1876    0.0075     0.0107         8561


## Second legs by leg-1 lead (§4B.3)

90-minute H/D/A of second legs by the home side's lead from leg 1, against
(a)'s mean prediction (which ignores the lead - it sees leg 1 only through Elo
and form). All European test seasons; the CL alone has too few.

In [16]:
legs2 = euro_test[euro_test["Leg"] == 2].copy()
legs2["lead"] = legs2["Leg1Lead"].clip(-2, 2).map({-2: "-2+", -1: "-1", 0: "0", 1: "+1", 2: "+2+"})
legs2[["pA", "pD", "pH"]] = proba[REFERENCE].loc[legs2.index][LABELS].to_numpy()
second_legs = legs2.groupby("lead").agg(
    n=("FTR", "size"),
    H=("FTR", lambda r: (r == "H").mean()), D=("FTR", lambda r: (r == "D").mean()), A=("FTR", lambda r: (r == "A").mean()),
    model_H=("pH", "mean"), model_D=("pD", "mean"), model_A=("pA", "mean"),
).reindex(["-2+", "-1", "0", "+1", "+2+"])
print(second_legs.round(3).to_string())

leg2_loss = {name: loss(name, legs2).mean() for name in [
    REFERENCE, "(b) domestic + Euro rows", "(b+leg2) with Leg2AggDiff", "(c) Euro-only LR", "(c+leg2) with Leg2AggDiff"]}
print("\nlog loss on these second legs:", {k: round(v, 4) for k, v in leg2_loss.items()})

       n      H      D      A  model_H  model_D  model_A
lead                                                    
-2+   42  0.476  0.167  0.357    0.370    0.243    0.387
-1    60  0.600  0.217  0.183    0.443    0.244    0.313
0     60  0.483  0.267  0.250    0.486    0.242    0.272
+1    41  0.488  0.268  0.244    0.558    0.237    0.205
+2+   45  0.467  0.156  0.378    0.587    0.228    0.185

log loss on these second legs: {'(a) domestic LR as is': np.float64(1.0082), '(b) domestic + Euro rows': np.float64(1.008), '(b+leg2) with Leg2AggDiff': np.float64(0.9909), '(c) Euro-only LR': np.float64(0.9995), '(c+leg2) with Leg2AggDiff': np.float64(0.9824)}


## League-phase matchday 8 (dead rubbers, reported only)

In [17]:
phase = cl_test.assign(md8=(cl_test["Stage"] == "LEAGUE_PHASE") & (cl_test["Round"] == "MD8"))
print(phase.groupby("md8").apply(lambda g: pd.Series({
    "n": len(g), "(a) log loss": loss(REFERENCE, g).mean(),
    "draw rate": (g["FTR"] == "D").mean()})).round(4).to_string())

           n  (a) log loss  draw rate
md8                                  
False  592.0        0.9340     0.1858
True    36.0        0.9069     0.1111


## Conclusions (numbers as run on 2026-11-05)

**Chosen: setup (c), an LR trained on European rows only** (CL + EL + ECL,
features `EloDiff`, `XGDDiffLast5`, `GoalsPerGameDiff`, `Comp_EL`, `Comp_ECL`,
`Knockout`; CL is the baseline competition), calibrated like the domestic
model, neutral venues scored both ways round.

- **Honest selection.** On the tune seasons (22-23/23-24) (c) has the best
  European log loss of the LR setups (0.9503, tied with (c+leg2); (a) 0.9521).
  On the report seasons (24-25/25-26) it beats (a) on the CL, -0.0167
  [-0.0286, -0.0051], 378 matches, and on CL+EL+ECL, -0.0116 [-0.0190, -0.0044],
  1,062 matches; better in both report seasons.
- **All four test seasons:** CL -0.0088 [-0.0177, +0.0001] (just short of
  *better* on 628 matches); CL+EL+ECL -0.0073 [-0.0129, -0.0018] (*better*).
  The gain is in 24-25/25-26: the new league-phase format, and more European
  seasons to train on.
- **(b)** (domestic + European rows): -0.0058 on the CL, CI crosses 0; better
  on the report seasons only. Domestic non-inferior (-0.00009 [-0.00056,
  +0.00037]), but (c) doesn't touch the domestic model at all.
- **Leg2AggDiff** doesn't earn its place: (c+leg2) - (c) = +0.0039 [-0.0009,
  +0.0091] on the CL, -0.0024 [-0.0055, +0.0006] on CL+EL+ECL. Leg 2 uses the
  leg-1 result only through Elo and form (the plan's default). The second-leg
  table still shows the pattern: a home side protecting a lead of 2+ wins only
  47% against the model's 59%, and one trailing by 1 wins 60% against 44%.
- **Model type stays LR (D5).** No tree passes §4B.8 against (c): best is
  (e) two-stage, -0.0104 [-0.0226, +0.0022] on the CL (3 of 4 seasons), +0.0004
  on CL+EL+ECL. (e) does beat (a) (-0.0192 [-0.0316, -0.0064]), but not the best
  LR. Nothing is added to requirements.txt.
- **Power.** Measured SD of paired differences: LR variant vs LR 0.083 (CL) /
  0.094 (CL+EL+ECL), plan assumed 0.069; tree vs LR 0.151 / 0.138, plan 0.150.
  Detecting 0.005 takes ~2,200-2,800 matches for LR variants (the plan said
  1,500) and ~6,000-7,000 for trees.